# Emotion Detection

Given an image of a facial expression, we want to determine what emotion is being portrayed. This is a multiclass classification problem as there are a total of 7 different emotions that we can classify instances as: angry, disgust, fear, happy, sad, surprised, and neutral. Our goal is to build a Convolutional Neural Network (CNN) model to classify these emotions from inputted images.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import pickle

## Load Preprocessed Data

In [ ]:
data = np.load('../data/processed_data.npz')

X_train = data['X_train']
X_test = data['X_test']
X_val = data['X_val']

X_train_normalized = data['X_train_normalized']
X_test_normalized = data['X_test_normalized']
X_val_normalized = data['X_val_normalized']

y_train = data['y_train']
y_test = data['y_test']
y_val = data['y_val']

y_train_cat = data['y_train_cat']
y_test_cat = data['y_test_cat']
y_val_cat = data['y_val_cat']

## Data Augmentation

There is a significant difference in the number of images across the different emotion classes. This could potentially lead to overfitting as our model will be trained using more images from certain classes than others. In order to get more data to use to train our model and to fix the imbalance, we will oversample the training data by performing image augmentation. We choose to oversample instead of undersample as undersampling can lead to the loss of important data. We also do not want to duplicate images as this can also lead to overfitting.

Perform data augmentation on all classes (not just minority ones) to create new versions of existing images. This teaches the model to handle small variations in images.

In [ ]:
# layer that randomly applies transformations (flips, rotations, zooms) to each input image during training only
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

## Class Weights

Use class weights to tell the loss function to pay more attention to less represented classes (like disgust and fear).

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))

## Helper Functions

In [ ]:
# plots accuracy and loss training curves
def plot_training_curves(history, title):
    plt.figure(figsize=(12, 5))

    # accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history['accuracy'], label='train')
    plt.plot(history['val_accuracy'], label='val')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy');
    plt.legend();
    
    # loss
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend();

    plt.show()

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# evaluates model, test accuracy & loss, confusion matrix, classification report
def evaluate_model(model, X_test, y_test, title):
    # test results
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f'{title} - Test Accuracy: {test_accuracy:.4f} Test Loss: {test_loss:.4f}')
    
    # confusion matrix
    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels).plot(cmap='Blues', xticks_rotation='vertical')
    plt.title(f'{title} - Confusion Matrix')
    plt.show()
    
    # classification report
    print(classification_report(y_true, y_pred, target_names=emotion_labels))
    

## Build Models


Build a Convolutional Neural Network (CNN) that classifies images by 1 of 7 emotion types.

### Model 1 - Baseline CNN

This model serves as the foundation for all later models to build on top of. It uses a single convolutional layer followed by batch normalization and max pooling to stabilize training and reduce overfitting.

In [ ]:
model1 = tf.keras.Sequential([
    # define input shape, 48x48 pixels, 1 channel (grayscale)
    layers.Input(shape=(48, 48, 1)),

    # convolutional layer
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    # convert 2D feature map into a 1D vector
    layers.Flatten(),

    # fully connected layer that learns combinations of the features detected by the convolutional layer, 64 neurons = 64 combinations/patterns
    layers.Dense(64, activation='relu'),

    # final layer with 7 neurons (one for each emotion)
    layers.Dense(7, activation='softmax')
])

model1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history1 = model1.fit(
    X_train_normalized, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_normalized, y_val_cat),
    callbacks=[early_stop]
)

# save history during training
with open('../history/history1.pkl', 'wb') as f:
    pickle.dump(history1.history, f)

model1.save('../models/model1.keras')

In [ ]:
plot_training_curves(history1, "Model 1")
evaluate_model(model1, X_test_normalized, y_test_cat, "Model 1")

The baseline model achieved 43% test accuracy. Training accuracy rose much higher than validation accuracy, indicating significant overfitting. Emotions with strong and easily distinguishable visual cues (Happy, Surprise) were easiest to identify correctly, while minority classes like Disgust were not predicted correctly at all. This model serves as a baseline for future improvements.

### Model 2 - Data Augmentation + Class Weights

This model extends the baseline by adding data augmentation (random flips, shifts, rotations) to increase variability and class weights to counter data imbalance by giving minority classes higher importance during training.

In [ ]:
model2 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(7, activation='softmax')
])

model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history2 = model2.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history2.pkl', 'wb') as f:
    pickle.dump(history2.history, f)

model2.save('../models/model2.keras')

In [ ]:
plot_training_curves(history2, "Model 2")
evaluate_model(model2, X_test, y_test_cat, "Model 2")

The model achieved 38% test accuracy, slightly lower than the baseline. Although overall accuracy decreased, predictions became more balanced across classes and Disgust, which the baseline never predicted correctly started being recognized. Applying class weights helped the model pay more attention to minority classes. While the model is now exposed to more variation, it still lacks the depth to learn complex facial features.

### Model 3 - Increased Network Depth

This model increases the network depth and number of filters used to expand feature extraction capacity and allow the model to learn more complex facial features. 

In [ ]:
model3 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history3 = model3.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('..history/history3.pkl', 'wb') as f:
    pickle.dump(history3.history, f)

model3.save('../models/model3c.keras')

In [ ]:
plot_training_curves(history3, "Model 3")
evaluate_model(model3, X_test, y_test_cat, "Model 3")

This model achieved 58% test accuracy. F1 scores increased across most emotions, specifically Angry, Disgust, and Fear. Classes with distinctive facial cues like Happy and Surprise remained the highest performing. Training and validation curves tracked closely indicating minimal overfitting. The confusion matrix also shows fewer misclassifications between visually similar emotions (Sad and Neutral).

### Model 4 - Regularization + Optimization Techniques

This model uses multiple regularization and optimization techniques to improve generalization and training stability:
- Learning rate scheduler for smoother and stabler convergence
- GlobalAveragePooling to reduce overall parameter count and improve spatial generalization
- Label smoothing to discourage overconfident predictions
- L2 regularization to penalize large weights and reduce overfitting

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

l2_strength = 1e-3

model4 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model4.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history4 = model4.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr]
)

with open('..history/history4.pkl', 'wb') as f:
    pickle.dump(history4.history, f)

model4.save('../models/model4.keras')

The model achieved 62% test accuracy and 1.234 test loss, which is the strongest performance so far among selected models. F1 scores improved for nearly all emotions, with the largest increases for Disgust (F1 = 0.46) and Fear (F1 = 0.40). Only Sad saw a minor decline. The confusion matrix shows reduced confusion among similar expressions, demonstrating better feature separation. Training was stable with smooth convergence and minimal overfitting.